# Probe Analysis

Train logistic-regression probes on JudgementLM attention activations and
evaluate at predicting valid/invalid labels.

**Labels** are the union of:
- `judgement_combined` — majority-vote LLM judge output
- `matching_status`    — whether the extraction matches a ground-truth row

**Analyses**
1. Baseline: raw judge model accuracy
2. Single-head selection: score every (layer, head) pair; visualise heatmap,
   PCA, calibration, probability distribution
3. Iterative greedy selection: sequentially add the head with the highest
   marginal contribution; same visualisations per iteration

In [ ]:
import sys
from pathlib import Path
%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'experiments'))
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
DATASET          = 'pond_ten'
EXTRACTION_MODEL = 'gemma-3-27b'
EXTRACTION_DATE  = '2026_04_14'
JUDGE_MODEL      = 'llama-3.1-8b'
TOP_K            = 5   # number of greedy iterations
N_FOLDS          = 5

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from analysis.loaders import (
    load_activations, load_combined_judgements,
    load_extraction, load_ground_truth, cached_match,
)
from analysis.plots import calibration_curve, probability_distribution
from scholarlm.utils.probe import (
    get_head_features, grouped_kfold_split,
    train_probe, eval_probe, eval_probe_detailed,
)
from scholarlm.utils.calibration import reliability_diagram_data
from scholarlm.utils.unit_conversion import apply_unit_conversion
from experiments.run_extraction import load_dataset_config
import paths

# Ensure figures directory exists
FIGURES_DIR = paths.figures_dir(DATASET)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
activations = load_activations(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE, JUDGE_MODEL)
judgements  = load_combined_judgements(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE)
judged_df   = pd.DataFrame(judgements)

config          = load_dataset_config(DATASET)
records         = load_extraction(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE)
extraction_df   = pd.DataFrame(records)
extraction_df   = apply_unit_conversion(extraction_df, config.unit_conversion_table)
ground_truth_df = load_ground_truth(config)

print(f'Activations:  {len(activations.files)} entries')
print(f'Judgements:   {len(judged_df)} records')
print(f'Extraction:   {len(extraction_df)} records')
print(f'Ground truth: {len(ground_truth_df)} records')

In [ ]:
# Load ground-truth matching from cache (produced by extraction_analysis.ipynb)
STRICT_MATCHING = {'title': 'title', 'attribute': 'attribute', 'converted_value': 'value'}
FUZZY_MATCHING  = {'name': 'name', 'location': 'location', 'ecosystem': 'ecosystem'}
cache_path = paths.extraction(DATASET, EXTRACTION_MODEL, EXTRACTION_DATE) / 'match_cache.pkl'

matching, _edges, _weights = cached_match(
    extraction_df, ground_truth_df,
    strict_matching=STRICT_MATCHING,
    fuzzy_matching=FUZZY_MATCHING,
    cache_path=cache_path,
)

# Index matching status by measurement_id string
matching_by_id = {}
for _gt_idx, ex_idx in matching:
    mid = str(extraction_df.iloc[ex_idx]['measurement_id'])
    matching_by_id[mid] = True

# Restrict to records that have activation entries
act_keys = set(activations.files)
records_with_act = (
    judged_df[judged_df['measurement_id'].astype(str).isin(act_keys)]
    .reset_index(drop=True)
)
measurement_ids = list(records_with_act['measurement_id'])
titles = records_with_act['title'].values
# Build page-level group key: (title, first_page)
# page_number is stored as a list, e.g. [1]; use first element as the page id
_page_nums = records_with_act['page_number'].apply(
    lambda v: v[0] if isinstance(v, list) and v else 0
).values
page_groups = np.array([f'{t}__p{p}' for t, p in zip(titles, _page_nums)])

# Labels = judgement_combined OR matching_status (union)
jlabels = records_with_act['judgement_combined'].values.astype(bool)
mlabels = np.array([matching_by_id.get(str(m), False) for m in measurement_ids])
labels  = jlabels | mlabels

print(f'n={len(labels)}  positive rate={labels.mean():.2%}')
print(f'  Judgement only: {(jlabels & ~mlabels).sum()}')
print(f'  Matching only:  {(~jlabels & mlabels).sum()}')
print(f'  Both:           {(jlabels & mlabels).sum()}')

## Baseline: Raw Judge Performance

In [ ]:
judge_preds = records_with_act['judgement_combined'].astype(bool).values

tp = int(( judge_preds &  labels).sum())
tn = int((~judge_preds & ~labels).sum())
fp = int(( judge_preds & ~labels).sum())
fn = int((~judge_preds &  labels).sum())
n  = len(labels)

acc  = (tp + tn) / n
prec = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
rec  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else float('nan')

print('Raw judge model performance (against union labels):')
print(f'  Accuracy:  {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall:    {rec:.4f}')
print(f'  F1:        {f1:.4f}')
print(f'  TP={tp}  TN={tn}  FP={fp}  FN={fn}')

In [ ]:
# Infer shape from first entry
_arr0 = np.array(activations[str(measurement_ids[0])], dtype=np.float32)
n_layers, n_heads, head_dim = _arr0.shape
print(f'Activation shape: n_layers={n_layers}, n_heads={n_heads}, head_dim={head_dim}')
print(f'Total heads: {n_layers * n_heads}')

# Load all activations into memory (one pass) then build per-head matrices
print('Loading activations into memory...')
_all_act = {
    str(mid): np.array(activations[str(mid)], dtype=np.float32)
    for mid in measurement_ids
}

print('Building per-head feature matrices...')
head_datasets: dict[tuple[int, int], np.ndarray] = {}
for l in range(n_layers):
    for h in range(n_heads):
        head_datasets[(l, h)] = np.stack(
            [_all_act[str(mid)][l, h, :] for mid in measurement_ids], axis=0
        )
del _all_act  # free memory
print(f'Done. Each head dataset shape: {head_datasets[(0, 0)].shape}')

## 1. Single-Head Selection

Score every (layer, head) pair with group-based 5-fold cross-validation,
then visualise the accuracy heatmap, PCA, calibration, and probability
distributions for the top head.

In [ ]:
def cv_score(X, y, groups, n_folds=N_FOLDS):
    """Return (mean_acc, fold_accs) using group-based k-fold CV."""
    fold_accs = []
    fold_test_idx = []
    fold_probs_list = []
    for train_idx, test_idx in grouped_kfold_split(groups, n_splits=n_folds):
        lr = LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', random_state=42)
        lr.fit(X[train_idx], y[train_idx])
        fold_accs.append(lr.score(X[test_idx], y[test_idx]))
        fold_test_idx.extend(test_idx)
        fold_probs_list.append(lr.predict_proba(X[test_idx])[:, 1])
    # Assemble out-of-fold probabilities in original sample order
    oof_probs = np.empty(len(y))
    sort_order = np.argsort(fold_test_idx)
    oof_probs[np.array(fold_test_idx)[sort_order]] = np.concatenate(fold_probs_list)[sort_order]
    return np.mean(fold_accs), list(fold_accs), oof_probs

print(f'Scoring {n_layers * n_heads} heads ({n_layers} layers x {n_heads} heads)...')
head_scores    = np.zeros((n_layers, n_heads))
head_fold_accs = {}
head_oof_probs = {}

for l in range(n_layers):
    for h in range(n_heads):
        mean_acc, fold_accs, oof_probs = cv_score(head_datasets[(l, h)], labels, page_groups)
        head_scores[l, h]     = mean_acc
        head_fold_accs[(l, h)] = fold_accs
        head_oof_probs[(l, h)] = oof_probs
    if (l + 1) % 8 == 0:
        print(f'  Layer {l + 1}/{n_layers} done')

best_l, best_h = np.unravel_index(head_scores.argmax(), head_scores.shape)
print(f'Best head: layer={best_l}, head={best_h}, acc={head_scores.max():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, n_heads * 0.38), max(4, n_layers * 0.38)))
im = ax.imshow(head_scores, vmin=0.5, vmax=1.0, cmap='viridis', aspect='auto')
# Mark best head
ax.add_patch(plt.Rectangle((best_h - 0.5, best_l - 0.5), 1, 1,
                            fill=False, edgecolor='red', lw=2))
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title(f'Single-head probe accuracy ({JUDGE_MODEL})')
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_single_head_heatmap.pdf', bbox_inches='tight')
fig

In [ ]:
X_best = head_datasets[(best_l, best_h)]
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_best)

fig, ax = plt.subplots(figsize=(5, 4))
for val, label_str, color in [(True, 'Valid', '#4ECDC4'), (False, 'Invalid', '#FF6B6B')]:
    mask = labels == val
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.35, s=12, label=label_str)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'PCA — best head (L{best_l}, H{best_h})')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_single_head_pca.pdf', bbox_inches='tight')
fig

In [ ]:
oof_probs_single = head_oof_probs[(best_l, best_h)]
diag_single = reliability_diagram_data(oof_probs_single, labels)
print(f'Single best head ECE = {diag_single["ece"]:.4f}')

fig = calibration_curve(diag_single, judge_labels=[f'L{best_l} H{best_h}'])
fig.axes[0].set_title(f'Calibration — best head (L{best_l}, H{best_h})')
fig.savefig(FIGURES_DIR / 'probe_single_head_calibration.pdf', bbox_inches='tight')
fig

In [ ]:
_df_single = pd.DataFrame({'prob': oof_probs_single, 'label': labels})
fig = probability_distribution(_df_single, prob_col='prob', label_col='label')
fig.axes[0].set_title(f'Prob distribution — best head (L{best_l}, H{best_h})')
fig.savefig(FIGURES_DIR / 'probe_single_head_prob_dist.pdf', bbox_inches='tight')
fig

In [ ]:
oof_preds_single = oof_probs_single > 0.5
tp_ = int(( oof_preds_single &  labels).sum())
tn_ = int((~oof_preds_single & ~labels).sum())
fp_ = int(( oof_preds_single & ~labels).sum())
fn_ = int((~oof_preds_single &  labels).sum())
n_  = len(labels)
acc_  = (tp_ + tn_) / n_
prec_ = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else float('nan')
rec_  = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else float('nan')
f1_   = 2 * prec_ * rec_ / (prec_ + rec_) if (prec_ + rec_) > 0 else float('nan')

print(f'=== Single-head probe summary (layer={best_l}, head={best_h}) ===')
print(f'  5-fold CV accuracy: {head_scores.max():.4f} '
      f'+/- {np.std(head_fold_accs[(best_l, best_h)]):.4f}')
print()
print('  Out-of-fold metrics (threshold=0.5):')
print(f'    Accuracy:  {acc_:.4f}')
print(f'    Precision: {prec_:.4f}')
print(f'    Recall:    {rec_:.4f}')
print(f'    F1:        {f1_:.4f}')
print(f'    TP={tp_}  TN={tn_}  FP={fp_}  FN={fn_}')

## 2. Iterative Greedy Head Selection

Sequentially add the attention head that maximally improves probe accuracy.
Iteration 1 reuses the single-head CV scores computed above.
Subsequent iterations evaluate each remaining head concatenated to the
current selected feature set.

In [ ]:
iteration_history = []
selected_heads    = []          # list of (layer, head) chosen so far
combined_X        = None        # growing feature matrix
remaining         = [(l, h) for l in range(n_layers) for h in range(n_heads)]

print(f'GREEDY FORWARD HEAD SELECTION  k={TOP_K}\n' + '=' * 60)

for iteration in range(TOP_K):
    print(f'\n--- Iteration {iteration + 1} ---')

    # Build the head accuracy matrix for this iteration
    acc_matrix  = np.full((n_layers, n_heads), np.nan)
    fold_per_head = {}
    oof_per_head  = {}

    for (l, h) in remaining:
        if combined_X is None:
            X_cand = head_datasets[(l, h)]
        else:
            X_cand = np.concatenate([combined_X, head_datasets[(l, h)]], axis=1)

        if iteration == 0:
            # Reuse single-head scores from the section above
            mean_acc   = head_scores[l, h]
            fold_accs  = head_fold_accs[(l, h)]
            oof_probs  = head_oof_probs[(l, h)]
        else:
            mean_acc, fold_accs, oof_probs = cv_score(X_cand, labels, page_groups)

        acc_matrix[l, h]   = mean_acc
        fold_per_head[(l, h)] = fold_accs
        oof_per_head[(l, h)]  = oof_probs

    # Select best remaining head
    valid_heads = [(l, h) for (l, h) in remaining if not np.isnan(acc_matrix[l, h])]
    best_lh     = max(valid_heads, key=lambda k: acc_matrix[k])
    bl, bh      = best_lh
    best_acc    = float(acc_matrix[bl, bh])
    best_folds  = fold_per_head[best_lh]
    best_oof    = oof_per_head[best_lh]
    print(f'  Selected: layer={bl:2d}, head={bh:2d}  acc={best_acc:.4f}')

    # Mark previously selected heads as NaN for visualisation
    vis_matrix = acc_matrix.copy()
    for sl, sh in selected_heads:
        vis_matrix[sl, sh] = np.nan

    # Update state
    X_best_head = head_datasets[best_lh]
    combined_X  = X_best_head.copy() if combined_X is None \
                  else np.concatenate([combined_X, X_best_head], axis=1)
    remaining.remove(best_lh)
    selected_heads.append(best_lh)

    iteration_history.append({
        'iteration':         iteration + 1,
        'selected_layer':    bl,
        'selected_head':     bh,
        'marginal_accuracy': best_acc,
        'fold_accuracies':   best_folds,
        'head_accuracy_matrix': vis_matrix,
        'combined_features': combined_X.copy(),
        'oof_probs':         best_oof,
        'oof_labels':        labels.copy(),
    })

print('\nDone.')

### Visualisation 1 — Head Accuracy Heatmaps per Iteration

In [ ]:
fig, axes = plt.subplots(1, TOP_K, figsize=(TOP_K * 4.2, 4))
if TOP_K == 1:
    axes = [axes]

for i, data in enumerate(iteration_history):
    ax = axes[i]
    mat = data['head_accuracy_matrix'].copy()

    # Show marginal contribution relative to previous iteration accuracy
    if i > 0:
        mat = mat - iteration_history[i - 1]['marginal_accuracy']

    # Sort heads within each layer for a cleaner display
    sorted_mat = np.sort(mat, axis=1)

    vmin, vmax = np.nanmin(sorted_mat), np.nanmax(sorted_mat)
    im = ax.imshow(sorted_mat, cmap='YlGn', aspect='auto', origin='lower',
                   vmin=vmin, vmax=vmax)
    bl, bh = data['selected_layer'], data['selected_head']
    ax.set_title(f'Iter {i + 1}\n(L{bl}, H{bh})', fontsize=9)
    ax.set_xlabel('Head (sorted)', fontsize=8)
    ax.set_ylabel('Layer', fontsize=8)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Accuracy' if i == 0 else 'Marginal Δ', fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_greedy_head_heatmaps.pdf', bbox_inches='tight')
fig

### Visualisation 2 — Accuracy Progression

In [ ]:
iters     = np.arange(1, TOP_K + 1)
mean_accs = [d['marginal_accuracy'] for d in iteration_history]
std_accs  = [np.std(d['fold_accuracies']) for d in iteration_history]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(iters, mean_accs, 'o-', lw=2, ms=7, color='#1b6ca8', label='Mean CV accuracy')
ax.fill_between(iters,
                np.array(mean_accs) - np.array(std_accs),
                np.array(mean_accs) + np.array(std_accs),
                alpha=0.2, color='#1b6ca8')
ax.axhline(head_scores.max(), ls='--', color='gray', label='Best single head')
ax.set_xlabel('Iteration')
ax.set_ylabel('CV Accuracy')
ax.set_title('Greedy head selection — accuracy progression')
ax.set_xticks(iters)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_greedy_accuracy_progression.pdf', bbox_inches='tight')
fig

### Visualisation 3 — Per-Fold Accuracy by Iteration

In [ ]:
fold_data = np.array([d['fold_accuracies'] for d in iteration_history])  # (TOP_K, N_FOLDS)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(fold_data, cmap='YlGn', aspect='auto',
               vmin=fold_data.min() - 0.01, vmax=fold_data.max())
ax.set_xticks(range(N_FOLDS))
ax.set_xticklabels([f'F{i}' for i in range(N_FOLDS)])
ax.set_yticks(range(TOP_K))
ax.set_yticklabels([f'Iter {i + 1}' for i in range(TOP_K)])
ax.set_xlabel('Fold')
ax.set_ylabel('Iteration')
ax.set_title('Per-fold accuracy by iteration')
for i in range(TOP_K):
    for j in range(N_FOLDS):
        ax.text(j, i, f'{fold_data[i, j]:.3f}', ha='center', va='center',
                fontsize=7, color='black')
plt.colorbar(im, ax=ax, label='Accuracy')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_greedy_fold_accuracy.pdf', bbox_inches='tight')
fig

### Visualisation 4 — PCA of Combined Features per Iteration

In [ ]:
fig, axes = plt.subplots(1, TOP_K, figsize=(TOP_K * 4, 3.8))
if TOP_K == 1:
    axes = [axes]

for i, data in enumerate(iteration_history):
    ax = axes[i]
    X_combined = data['combined_features']
    true_labels = data['oof_labels']

    pca_i = PCA(n_components=2)
    Xp = pca_i.fit_transform(X_combined)

    for val, lbl, color in [(True, 'Valid', '#4ECDC4'), (False, 'Invalid', '#FF6B6B')]:
        mask = true_labels == val
        ax.scatter(Xp[mask, 0], Xp[mask, 1], c=color, alpha=0.35, s=8, label=lbl)

    dims = X_combined.shape[1]
    ax.set_title(f'Iter {i + 1} ({dims}d)', fontsize=9)
    ax.set_xlabel(f'PC1 ({pca_i.explained_variance_ratio_[0]:.1%})', fontsize=7)
    ax.set_ylabel(f'PC2 ({pca_i.explained_variance_ratio_[1]:.1%})', fontsize=7)
    if i == 0:
        ax.legend(fontsize=7)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_greedy_pca.pdf', bbox_inches='tight')
fig

### Visualisation 5 — Calibration Curves per Iteration

In [ ]:
colors_iter = plt.cm.tab10(np.linspace(0, 0.9, TOP_K))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Perfect calibration')

for i, data in enumerate(iteration_history):
    diag = reliability_diagram_data(data['oof_probs'], data['oof_labels'])
    valid = ~np.isnan(diag['bin_accuracy'])
    ax.plot(
        diag['bin_centers'][valid],
        diag['bin_accuracy'][valid],
        'o-', color=colors_iter[i], lw=1.8, ms=5,
        label=f"Iter {i + 1} (ECE={diag['ece']:.3f})",
    )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed frequency')
ax.set_title('Calibration curves — iterative head selection')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'probe_greedy_calibration.pdf', bbox_inches='tight')
fig

### Visualisation 6 — Probability Distributions (Final Iteration)

In [ ]:
final = iteration_history[-1]
_df_final = pd.DataFrame({'prob': final['oof_probs'], 'label': final['oof_labels']})
fig = probability_distribution(_df_final, prob_col='prob', label_col='label')
fig.axes[0].set_title(f'Predicted probability — {TOP_K}-head ensemble')
fig.savefig(FIGURES_DIR / 'probe_greedy_prob_dist.pdf', bbox_inches='tight')
fig

### Summary

In [ ]:
print('=' * 60)
print('GREEDY SELECTION SUMMARY')
print('=' * 60)
for data in iteration_history:
    bl = data['selected_layer']
    bh = data['selected_head']
    acc = data['marginal_accuracy']
    std = np.std(data['fold_accuracies'])
    print(f"  Iter {data['iteration']}: L{bl:2d} H{bh:2d}  acc={acc:.4f} +/- {std:.4f}")

print()
print('=' * 60)
print('OUT-OF-FOLD METRICS BY ITERATION  (threshold=0.5)')
print('=' * 60)
print(f'{"Iter":>4}  {"Accuracy":>8}  {"Precision":>9}  {"Recall":>6}  {"F1":>6}  ECE')

for data in iteration_history:
    probs = data['oof_probs']
    yl    = data['oof_labels']
    preds = probs > 0.5
    tp_ = int(( preds &  yl).sum())
    tn_ = int((~preds & ~yl).sum())
    fp_ = int(( preds & ~yl).sum())
    fn_ = int((~preds &  yl).sum())
    n_  = len(yl)
    acc_  = (tp_ + tn_) / n_
    prec_ = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else float('nan')
    rec_  = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else float('nan')
    f1_   = 2 * prec_ * rec_ / (prec_ + rec_) if (prec_ + rec_) > 0 else float('nan')
    diag  = reliability_diagram_data(probs, yl)
    print(f"  {data['iteration']:>2d}   {acc_:>8.4f}  {prec_:>9.4f}  "
          f"{rec_:>6.4f}  {f1_:>6.4f}  {diag['ece']:.4f}")